# Ноутбук 02 — Стационарность, АКФ, ЧАКФ, АДФ-тест
**Подраздел 2.3 ПЗ — Стационарность и корреляционные свойства ряда**

Запускать после `01_preprocessing_features.ipynb`.

Артефакты:
- `reports/figures/fig_2_8_acf_original.png`  — Рисунок 2.8
- `reports/figures/fig_2_9_pacf_original.png` — Рисунок 2.9
- `reports/tables/table_2_2_adf_original.csv` — Таблица 2.2
- `data/interim/weekly_diff1.parquet`          — дифференцированный недельный ряд
- `reports/figures/fig_2_10_acf_diff1.png`    — Рисунок 2.10
- `reports/figures/fig_2_11_pacf_diff1.png`   — Рисунок 2.11
- `reports/tables/table_2_3_seasonal_acf.csv` — Подтверждение S = 52

## Ячейка 0 — Импорты и конфигурация

In [ ]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.config import (
    DATA_INT, FIGURES, TABLES,
    SEASONAL_PERIOD,
)

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# --- Константы анализа ---
# LAGS_ACF_ORIG=156: глубина для исходного ряда — показывает медленное затухание (п. 2.3.1).
# LAGS_ACF=60: глубина <= T/4 = 242//4 = 60 — надёжная зона для diff1 (п. 2.3.4).
#   При T=242 расчёт на лагах > T/4 статистически ненадёжен:
#   доверительные границы Бартлетта 1.96/sqrt(n_k) расширяются.
# SEASONAL_LAGS=[52,104]: лаг 156 исключён — при T=242 доступно
#   лишь 86 наблюдательных пар, граница Бартлетта слишком широка.
LAGS_ACF_ORIG   = 156
LAGS_ACF        = 60               # <= T/4 для diff1
ALPHA           = 0.05             # уровень значимости для ДИ АКФ и АДФ-теста
# Спецификация АДФ-теста: константа + тренд (п. 2.3.3)
# Исключает ложное принятие H0 из-за детерминированных компонент
ADF_REGRESSION  = "ct"
SEASON          = SEASONAL_PERIOD          # 52 недели (из config.py)
# Лаг 156 исключён из аннотации: при T=242 доступно лишь 86 пар;
# доверительные границы Бартлетта слишком широки для надёжного вывода.
SEASONAL_LAGS   = [SEASON, 2 * SEASON]    # [52, 104]

# --- Настройки оформления графиков ---
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print(f"SEASONAL_PERIOD  : {SEASON}")
print(f"LAGS_ACF_ORIG    : {LAGS_ACF_ORIG}")
print(f"LAGS_ACF (diff1) : {LAGS_ACF}")
print(f"ADF_REGRESSION   : {ADF_REGRESSION}")
print(f"SEASONAL_LAGS    : {SEASONAL_LAGS}")

## Ячейка 1 — Загрузка недельного ряда

Источник: `data/interim/weekly_sales.parquet` (артефакт Ячейки 7 ноутбука 01).

Индекс приводится к частоте `W-MON`.
Поскольку ряд формируется агрегацией дневных продаж, пропуски в индексе
возможны для нерабочих недель; они заполняются методом `pad` (forward-fill).
После заполнения ноутбук прерывается с ошибкой, если в ряде остались NaN.

In [ ]:
_raw = pd.read_parquet(DATA_INT / "weekly_sales.parquet")

# Столбец может называться 'sales' или быть единственным столбцом DataFrame
weekly = _raw.iloc[:, 0] if isinstance(_raw, pd.DataFrame) else _raw
weekly.index = pd.to_datetime(weekly.index)
weekly.name = "sales"

# Привод к явной частоте W-MON; пропуски заполняются pad (forward-fill)
weekly = weekly.asfreq("W-MON")
weekly = weekly.fillna(method="pad")

n_gaps = weekly.isna().sum()
assert n_gaps == 0, (
    f"После заполнения в weekly_sales остались пропуски: {n_gaps}. "
    "Запустите ноутбук 01 заново."
)

print(f"Наблюдений : {len(weekly)}")
print(f"Период     : {weekly.index[0].date()} — {weekly.index[-1].date()}")
print(f"Пропусков  : {n_gaps}  — OK")
print(f"Мин        : {weekly.min():,.0f}")
print(f"Макс       : {weekly.max():,.0f}")

## Вспомогательная функция построения АКФ / ЧАКФ

**Ограничение ЧАКФ:**  
`statsmodels.tsa.stattools.pacf` требует `nlags < len(series) // 2`.  
Вследствие этого для `kind='pacf'` глубина лагов автоматически ограничивается
значением `min(lags, len(series) // 2 - 1)`.  
Для `kind='acf'` ограничение не применяется — `acf` допускает глубину до `len(series) - 1`.

In [ ]:
def plot_correlation_function(
    series, lags, kind, title, fig_path,
    seasonal_lags=None, alpha=0.05
):
    """
    Строит и сохраняет АКФ или ЧАКФ.

    Parameters
    ----------
    series       : pd.Series — временной ряд
    lags         : int       — максимальная глубина лагов (для АКФ)
    kind         : str       — 'acf' или 'pacf'
    title        : str       — заголовок графика
    fig_path     : Path      — путь для сохранения PNG
    seasonal_lags: list[int] — лаги, на которых ожидаются сезонные пики
    alpha        : float     — уровень значимости для ДИ Бартлетта

    Примечание
    ----------
    Для ЧАКФ эффективная глубина lags_eff = min(lags, len(series) // 2 - 1),
    поскольку statsmodels требует nlags < len(series) // 2.
    Сезонные аннотации отображаются только для лагов <= lags_eff.
    """
    fig, ax = plt.subplots(figsize=(12, 4))

    if kind == "acf":
        lags_eff = lags
        plot_acf(series, lags=lags_eff, alpha=alpha, ax=ax, zero=False)
    elif kind == "pacf":
        # statsmodels: nlags must be < len(series) // 2
        max_pacf_lags = len(series) // 2 - 1
        lags_eff = min(lags, max_pacf_lags)
        if lags_eff < lags:
            print(
                f"[ЧАКФ] Запрошено лагов: {lags}. "
                f"Ограничено до {lags_eff} (< len(series) // 2 = {len(series) // 2})."
            )
        plot_pacf(series, lags=lags_eff, alpha=alpha, ax=ax, zero=False, method="ywm")
    else:
        raise ValueError(f"kind должен быть 'acf' или 'pacf', получено: {kind!r}")

    ax.set_title(title)
    ax.set_xlabel("Лаг (недели)")
    ax.set_ylabel("Коэффициент корреляции")

    # Аннотация сезонных пиков (п. 2.3.1 / 2.3.5)
    if seasonal_lags:
        for lag in seasonal_lags:
            if lag <= lags_eff:
                ax.axvline(x=lag, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
                ax.annotate(
                    f"lag={lag}",
                    xy=(lag, ax.get_ylim()[1] * 0.85),
                    fontsize=8, color="red", ha="center"
                )

    plt.tight_layout()
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Рисунок сохранён: {fig_path}")

print("Вспомогательная функция определена.")

## Ячейка 2 — АКФ исходного ряда (Рисунок 2.8)

**Ожидаемые результаты (п. 2.3.1):**
- Медленное, практически линейное затухание АКФ — признак трендовой нестационарности.
- Выраженные пики на лагах 52 и 104 — сезонная структура с периодом S = 52.

**Примечание (п. 2.3.5):**
Анализ проводится на агрегированном недельном ряде (T = 242).
Пик на лаге 52 статистически значим. Пик на лаге 104 рассчитывается по 138 парам —
значимость снижается, но пик используется как подтверждающий. Лаг 156 (86 пар)
исключён из аннотации: доверительные границы Бартлетта слишком широки для надёжного
вывода о значимости.

In [ ]:
plot_correlation_function(
    weekly,
    lags=LAGS_ACF_ORIG,          # 156 — показывает медленное затухание на всём диапазоне
    kind="acf",
    title="Рисунок 2.8 — АКФ исходного недельного ряда суммарных продаж (лаги 1–156)",
    fig_path=FIGURES / "fig_2_8_acf_original.png",
    seasonal_lags=SEASONAL_LAGS,  # аннотируем [52, 104]; лаг 156 исключён
    alpha=ALPHA,
)

## Ячейка 3 — ЧАКФ исходного ряда (Рисунок 2.9)

In [ ]:
plot_correlation_function(
    weekly,
    lags=LAGS_ACF_ORIG,
    kind="pacf",
    title="Рисунок 2.9 — ЧАКФ исходного недельного ряда суммарных продаж",
    fig_path=FIGURES / "fig_2_9_pacf_original.png",
    seasonal_lags=SEASONAL_LAGS,
    alpha=ALPHA,
)

## Ячейка 4 — АДФ-тест исходного ряда (Таблица 2.2)

**Спецификация (п. 2.3.3):** `regression='ct'` (константа + тренд).  
Поскольку СТЛ-декомпозиция зафиксировала восходящий тренд и структурные сдвиги
(землетрясение 2016, НДС 2014), спецификация с трендом исключает ложное принятие
H0 из-за детерминированных компонент.

In [ ]:
adf_result = adfuller(weekly, regression=ADF_REGRESSION)

adf_stat   = adf_result[0]
adf_pvalue = adf_result[1]
adf_crit   = adf_result[4]   # критические значения

# Сохраняем таблицу 2.2
table_22 = pd.DataFrame({
    "Показатель": [
        "Тестовая статистика",
        "p-значение",
        "Критическое значение 1%",
        "Критическое значение 5%",
        "Критическое значение 10%",
        "Вывод",
    ],
    "Значение": [
        f"{adf_stat:.4f}",
        f"{adf_pvalue:.4f}",
        f"{adf_crit['1%']:.4f}",
        f"{adf_crit['5%']:.4f}",
        f"{adf_crit['10%']:.4f}",
        "H0 не отвергается, ряд нестационарен" if adf_pvalue > ALPHA else "H0 отвергается, ряд стационарен",
    ],
})
table_22.to_csv(TABLES / "table_2_2_adf_original.csv", index=False)

print("Таблица 2.2 — Результаты АДФ-теста для исходного ряда")
print(table_22.to_string(index=False))

## Ячейка 5 — Определение порядка d и дифференцирование (п. 2.3.3)

In [ ]:
# Параметр d: 1 при p > alpha, 0 иначе
d = 1 if adf_pvalue > ALPHA else 0
print(f"p-значение АДФ = {adf_pvalue:.4f} -> d = {d}")

# Первое дифференцирование
diff1 = weekly.diff(1).dropna()
diff1.name = "sales_diff1"

diff1.to_frame().to_parquet(DATA_INT / "weekly_diff1.parquet")
print(f"Ряд diff1: {len(diff1)} наблюдений")
print(f"Пропусков: {diff1.isna().sum()}")

## Ячейка 6 — АКФ дифференцированного ряда (Рисунок 2.10)

**Глубина лагов (п. 2.3.4):** после дифференцирования T_eff = T - 1 = 241.
Глубина ограничена LAGS_ACF = 60 (<=T/4), поскольку расчёт на лагах > T/4
статистически ненадёжен из-за малого числа наблюдательных пар.

In [ ]:
plot_correlation_function(
    diff1,
    lags=LAGS_ACF,               # 60 — надёжная зона при T_eff=241
    kind="acf",
    title="Рисунок 2.10 — АКФ ряда после первого дифференцирования (лаги 1–60)",
    fig_path=FIGURES / "fig_2_10_acf_diff1.png",
    seasonal_lags=SEASONAL_LAGS,  # [52, 104]; лаг 52 < 60 — аннотируется
    alpha=ALPHA,
)

## Ячейка 7 — ЧАКФ дифференцированного ряда (Рисунок 2.11)

In [ ]:
plot_correlation_function(
    diff1,
    lags=LAGS_ACF,
    kind="pacf",
    title="Рисунок 2.11 — ЧАКФ ряда после первого дифференцирования",
    fig_path=FIGURES / "fig_2_11_pacf_diff1.png",
    seasonal_lags=SEASONAL_LAGS,
    alpha=ALPHA,
)

## Ячейка 12 — Статистическое подтверждение S = 52 (Таблица 2.3)

**Корневая причина исправления:**  
Предыдущая версия вычисляла `acf()` на нестационарном ряде `weekly` и применяла
единую границу Бартлетта `1.96/sqrt(N)`, как для белого шума. Вследствие этого
на лагах 104 и 156, где число пар мало, `|acf_vals[lag]|` оказывался ниже границы
при наличии визуального пика, и `assert` падал ложно.

**Исправление:**
- АКФ вычисляется на стационарном ряде `diff1` (п. 2.3.3: d = 1).
- Граница Бартлетта рассчитывается индивидуально для каждого лага k:
  `bound_k = 1.96 / sqrt(n - k)`, где n = len(diff1).
- Лаг 156 исключён из таблицы: при T = 242 доступно лишь 86 пар.
- `assert` проверяет только лаг 52 (наиболее надёжный).

In [ ]:
# Исправление: тест на дифференцированном ряде diff1, а не на weekly
n = len(diff1)                                  # 241
acf_vals, acf_confint = acf(diff1, nlags=LAGS_ACF_ORIG, alpha=ALPHA, fft=True)

# Индивидуальная граница Бартлетта для каждого лага k: 1.96/sqrt(n-k)
rows = []
for lag in SEASONAL_LAGS:                       # [52, 104] — лаг 156 исключён
    val   = acf_vals[lag]
    n_k   = n - lag                             # реальное число пар
    bound = 1.96 / np.sqrt(n_k)                 # индивидуальная граница
    is_sig = abs(val) > bound
    rows.append({
        "Лаг (недели)": lag,
        "ACF": round(val, 4),
        "n пар": n_k,
        "Граница (скорр.)": round(bound, 4),
        "Значим (p=0.05)": "Да" if is_sig else "Нет",
    })

table_23 = pd.DataFrame(rows)
table_23.to_csv(TABLES / "table_2_3_seasonal_acf.csv", index=False)

print("Таблица 2.3 — Сезонные пики АКФ дифференцированного ряда")
print(table_23.to_string(index=False))
print()
print("Примечание: лаг 156 не включён в таблицу — при T=242 "
      "доступно лишь 86 наблюдательных пар; "
      "доверительные границы не позволяют сделать надёжный вывод.")

# assert: проверяем только лаг 52 — наиболее надёжный (п. 2.3.5)
n_k_52 = n - SEASON
assert abs(acf_vals[SEASON]) > 1.96 / np.sqrt(n_k_52), (
    f"Сезонный пик на лаге {SEASON} не подтверждён на дифференцированном ряде. "
    f"ACF={acf_vals[SEASON]:.4f}, граница={1.96/np.sqrt(n_k_52):.4f}. "
    "Пересмотрите S."
)
print(f"OK: сезонный пик на лаге {SEASON} подтверждён (p=0.05).")